# W-V: frozen V-JEPA 2 on the QG world (Kaggle GPU)

**What it does.** Installs `weather-world-model`, regenerates the stationary QG data of step 2 with the
runner's own functions and data seed (profile `full`: 80 runs, split 56/12/12 by run, spin-up 12 000 steps,
100 snapshots per run), renders the upper-layer winds as video (see `weather_world_model.worlds.vjepa2_qg`
for the fixed rendering), encodes it with the frozen V-JEPA 2 ViT-L 256 (`facebook/vjepa2-vitl-fpc64-256`,
pinned revision), and measures W-P (ridge readouts of the seven QG diagnostics, plus a random-weight
control) and W-C (arrow-of-time probe, plain reversal and the mirrored control). It writes
`/kaggle/working/qg-vjepa2.json` with the code commit, model id and revision, configuration, values, and the
W-V comparison against the small-JEPA records shipped with the code. Only the frozen variant is measured.

**Settings.** Accelerator: GPU T4 x1 (or P100). Internet: on (weights from Hugging Face, about 1.3 GB).
Attach the private dataset `weather-world-model-wheel` for install mode `wheel` (see `docs/KAGGLE.md`).
Then *Run all*.

**Expected runtime (an estimate, not measured on a GPU).** About 10 min to simulate the QG data on the
Kaggle CPU. Then about 3 400 clips of 32 video frames through ViT-L (2 800 frozen, 560 for the random-weight
control), 4.1 TFLOP each (counted with `torch.utils.flop_counter`), 14 PFLOP in all: 10 to 30 min on a T4 in
float16 (10 to 25 TFLOP/s sustained), 30 to 60 min on a P100. Total: well under 1.5 h on a T4.
**GPU memory:** about 2 to 4 GB with batch size 4 (float16 weights 0.65 GB); the record stores the peak.
**RAM:** about 4 GB.

**Result.** When the run ends, open the *Output* panel on the right (or *Save Version* and open the
version's *Output* tab) and download `qg-vjepa2.json`; it goes to `results/qg/` in the repository.

Dry run: set `DRY_RUN = True` to check the pipeline in a few minutes on CPU with the smoke profile and a
tiny randomly initialised V-JEPA 2 (no download); its values mean nothing.

In [ ]:
# ---- settings ---------------------------------------------------------------
INSTALL_MODE = "wheel"  # "wheel": private Kaggle dataset; "git": public GitHub repository
GIT_COMMIT = "main"  # mode "git" only: the commit (full SHA) to install and to read records from
WHEEL_DIR = "/kaggle/input/weather-world-model-wheel"
DRY_RUN = False  # True: smoke profile + tiny random model on CPU, to test the notebook
BATCH_SIZE = 4  # clips per forward pass (32 frames each)
RANDOM_CONTROL = True  # W-P control: same architecture, random weights (adds ~20 % runtime)
TRANSFORMERS_VERSION = "5.24.1"
OUT = "/kaggle/working/qg-vjepa2.json"

In [ ]:
import glob
import json
import os
import subprocess
import sys

assert sys.version_info >= (3, 11), sys.version
# The package declares Python 3.12; its code runs on 3.11, which some Kaggle images still use.
# --no-deps: numpy and torch are already in the Kaggle image (CUDA build).
pip = [sys.executable, "-m", "pip", "install", "-q", "--ignore-requires-python", "--no-deps"]
if INSTALL_MODE == "wheel":
    wheels = sorted(glob.glob(f"{WHEEL_DIR}/*.whl"))
    assert len(wheels) == 1, f"expected one wheel in {WHEEL_DIR}, found {wheels}"
    subprocess.run([*pip, "--force-reinstall", wheels[0]], check=True)
    with open(f"{WHEEL_DIR}/build-info.json") as f:
        BUILD = json.load(f)
    RECORDS = sorted(glob.glob(f"{WHEEL_DIR}/records/qg-stationary*.json"))
elif INSTALL_MODE == "git":
    url = "https://github.com/gabrielkpo/weather-world-model"
    subprocess.run([*pip, "--force-reinstall", f"git+{url}@{GIT_COMMIT}"], check=True)
    subprocess.run(["git", "clone", "-q", url, "/kaggle/working/wwm-src"], check=True)
    subprocess.run(
        ["git", "-C", "/kaggle/working/wwm-src", "checkout", "-q", GIT_COMMIT], check=True
    )
    commit = subprocess.run(
        ["git", "-C", "/kaggle/working/wwm-src", "rev-parse", "HEAD"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
    BUILD = {"code_commit": commit, "source": url}
    RECORDS = sorted(glob.glob("/kaggle/working/wwm-src/results/qg/qg-stationary*.json"))
else:
    raise ValueError(INSTALL_MODE)
RECORDS = [r for r in RECORDS if not r.endswith(".partial.json")]
subprocess.run([*pip[:5], f"transformers=={TRANSFORMERS_VERSION}"], check=True)
print(BUILD)
print("records:", [os.path.basename(r) for r in RECORDS])

In [ ]:
import time
from dataclasses import asdict
from datetime import UTC, datetime
from pathlib import Path

import numpy as np
import torch
import transformers
from transformers import VJEPA2Config, VJEPA2Model

from weather_world_model.worlds import run_qg
from weather_world_model.worlds import vjepa2_qg as vq
from weather_world_model.worlds.qg import DIAGNOSTIC_NAMES, QGConfig

DEVICE = "cuda" if torch.cuda.is_available() and not DRY_RUN else "cpu"
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print(
    sys.version, "torch", torch.__version__, "transformers", transformers.__version__, DEVICE, GPU
)
assert DRY_RUN or DEVICE == "cuda", "enable a GPU accelerator (Settings > Accelerator)"
T0 = time.perf_counter()
TIMES = {}

In [ ]:
# ---- data: the runner's own simulation, profile and data seed ---------------------
profile_name = "smoke" if DRY_RUN else "full"
profile = run_qg.PROFILES[profile_name]
t = time.perf_counter()
data = run_qg.simulate(profile)
TIMES["simulate_seconds"] = time.perf_counter() - t
stationarity = run_qg.stationarity_check(
    data.diags["train"][..., DIAGNOSTIC_NAMES.index("kinetic_energy")],
    profile.stationarity_tolerance,
)
print(f"simulated in {TIMES['simulate_seconds']:.0f}s", stationarity)
assert stationarity["passes"] is not False, "training data not stationary (protocol amendment 5)"
fields = {k: vq.physical(v, data.mean, data.std).astype(np.float32) for k, v in data.frames.items()}
diags = data.diags
print({k: v.shape for k, v in fields.items()})

In [ ]:
# ---- the comparison records, and a check that the data match them -----------------
records = {Path(p).stem: json.loads(Path(p).read_text()) for p in RECORDS}
data_check = {}
for name, rec in records.items():
    same_config = rec.get("config", {}).get("data_seed") == profile.data_seed and rec.get(
        "split_runs"
    ) == dict(zip(("train", "val", "test"), run_qg.split_sizes(profile.n_runs), strict=True))
    ref = rec.get("stationarity", {})
    data_check[name] = {
        "same_seed_and_splits": bool(same_config),
        "record_kinetic_energy_first_third": ref.get("first_third_mean"),
        "this_run_kinetic_energy_first_third": stationarity["first_third_mean"],
    }
print(json.dumps(data_check, indent=1))
# The simulation is chaotic: another CPU can round differently and give other snapshots with the
# same statistics. Same seed and splits is what is required; the energies are reported.

In [ ]:
# ---- V-JEPA 2 -----------------------------------------------------------------------
MODEL_ID, REVISION = vq.MODEL_ID, vq.MODEL_REVISION
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32  # T4/P100 have no bfloat16
if DRY_RUN:
    tiny = dict(
        hidden_size=64,
        num_attention_heads=4,
        num_hidden_layers=2,
        pred_hidden_size=32,
        pred_num_attention_heads=4,
        pred_num_hidden_layers=1,
    )
    torch.manual_seed(0)
    model = VJEPA2Model(VJEPA2Config(**tiny))
else:
    model = VJEPA2Model.from_pretrained(MODEL_ID, revision=REVISION, dtype=DTYPE)
model = model.to(DEVICE).eval()
cfg = model.config
SPEC = vq.VideoSpec(
    image_size=cfg.crop_size,
    patch_size=cfg.patch_size,
    tubelet=cfg.tubelet_size,
    clip_len=profile.clip_len,
    clip_stride=profile.clip_stride,
)


def make_encoder(m):
    def encode(video):
        return m.get_vision_features(video.to(device=DEVICE, dtype=DTYPE))

    return encode


encoder = make_encoder(model)
with torch.no_grad():
    probe_clip = torch.as_tensor(fields["train"][:1, : SPEC.clip_len], device=DEVICE)
    scales = vq.fit_render_scales(fields["train"])
    t = time.perf_counter()
    z = vq.encode_clips(encoder, probe_clip, scales, SPEC)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    print("one clip:", tuple(z.shape), f"{time.perf_counter() - t:.2f}s", "(includes warm-up)")
assert z.shape == (1, SPEC.clip_len, cfg.hidden_size)

In [ ]:
# ---- W-P and W-C on frozen features --------------------------------------------------
def log(msg):
    print(f"[{time.perf_counter() - T0:7.0f}s] {msg}", flush=True)


t = time.perf_counter()
frozen = vq.evaluate_encoder(
    encoder, fields, diags, SPEC, batch_size=BATCH_SIZE, device=DEVICE, log=log
)
TIMES["frozen_seconds"] = time.perf_counter() - t
print(json.dumps({k: v for k, v in frozen.items() if k.startswith("arrow")}, indent=1))
print({q: round(s["test_r2"], 3) for q, s in frozen["readouts"].items()})

In [ ]:
# ---- W-P control: same architecture, random weights (seed 0) -------------------------
random_control = None
if RANDOM_CONTROL:
    torch.manual_seed(0)
    rnd = VJEPA2Model(model.config).to(device=DEVICE, dtype=DTYPE).eval()
    t = time.perf_counter()
    random_control = vq.evaluate_encoder(
        make_encoder(rnd),
        fields,
        diags,
        SPEC,
        batch_size=BATCH_SIZE,
        device=DEVICE,
        arrow=False,
        log=log,
    )
    TIMES["random_control_seconds"] = time.perf_counter() - t
    print({q: round(s["test_r2"], 3) for q, s in random_control["readouts"].items()})
    del rnd

In [ ]:
# ---- W-V comparison and record ---------------------------------------------------------
refs = {name: vq.small_jepa_reference(rec) for name, rec in records.items()}
norms = {name: vq.record_norm(rec, Path(name + ".json")) for name, rec in records.items()}
wv = vq.wv_comparison(frozen, refs, norms) if refs else {"note": "no comparison record found"}
record = {
    "code_commit": BUILD.get("code_commit"),
    "install_mode": INSTALL_MODE,
    "date": datetime.now(UTC).isoformat(timespec="seconds"),
    "dry_run": DRY_RUN,
    "model_id": None if DRY_RUN else MODEL_ID,
    "model_revision": None if DRY_RUN else REVISION,
    "variant": "frozen",
    "profile": profile_name,
    "config": {
        **asdict(profile),
        "video": asdict(SPEC),
        "batch_size": BATCH_SIZE,
        "dtype": str(DTYPE),
        "probe_seeds": list(vq.PROBE_SEEDS),
    },
    "qg_config": asdict(QGConfig()),
    "split_runs": dict(
        zip(("train", "val", "test"), run_qg.split_sizes(profile.n_runs), strict=True)
    ),
    "diagnostic_names": list(DIAGNOSTIC_NAMES),
    "stationarity": stationarity,
    "data_check": data_check,
    "environment": {
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "device": DEVICE,
        "gpu": GPU,
        "peak_gpu_memory_gb": torch.cuda.max_memory_allocated() / 2**30
        if DEVICE == "cuda"
        else None,
    },
    "seconds": {**TIMES, "wall_seconds": time.perf_counter() - T0},
    "frozen": frozen,
    "random_control": random_control,
    "W-V": wv,
}
Path(OUT).write_text(json.dumps(record, indent=2) + "\n")
print(json.dumps(wv, indent=1))
print("written", OUT, f"{os.path.getsize(OUT) / 1024:.0f} kB")